<a href="https://colab.research.google.com/github/sharniks/langchain-fundamentals/blob/main/tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') # Make sure you've added your HF_TOKEN to Colab secrets!

In [42]:
!pip install -q langchain-huggingface langchain-core requests

In [43]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [44]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
    """Given 2 number a and b this tool return their product"""
    return a * b

In [45]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [46]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 number a and b this tool return their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [47]:
#tool binding

In [48]:
llm = HuggingFaceEndpoint(
    repo_id='openai/gpt-oss-20b',
    task='text-generation'
)

model = ChatHuggingFace(llm=llm)

In [49]:
llm_with_tools = model.bind_tools([multiply])

In [50]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 133, 'total_tokens': 190}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f7bba-3045-7e51-9f47-677a22780937-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 57, 'total_tokens': 190})

In [51]:
query = HumanMessage('can you multiply 3 with 1000')

In [52]:
messages = [query]

In [53]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [54]:
result = llm_with_tools.invoke(messages)

In [55]:
messages.append(result)

In [56]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='{"a":3,"b":1000}', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-ac53824fd7d9e81f', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 138, 'total_tokens': 183}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f7bba-33ba-7543-910e-4ef62e49169e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-ac53824fd7d9e81f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 45, 'total_tokens': 183})]

In [57]:
result.tool_calls[0]['args']

{'a': 3, 'b': 1000}

In [58]:
tool_result = multiply.invoke(result.tool_calls[0])

In [59]:
messages.append(tool_result)

In [60]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='{"a":3,"b":1000}', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a":3,"b":1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-ac53824fd7d9e81f', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 138, 'total_tokens': 183}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f7bba-33ba-7543-910e-4ef62e49169e-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-ac53824fd7d9e81f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 138, 'output_tokens': 45, 'total_tokens': 183}),
 ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-ac53824fd7d9e81f')]

In [61]:
llm_with_tools.invoke(messages).content

'The product of 3 and 1000 is **3000**.'